In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History
import pandas as pd
import numpy as np

save_path = './comparison_forms/'

db_url = "postgresql://tongli1997@db.pcic.uvic.ca:5433/crmp?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [3]:
station_name = ['Judge Creek', 'Rithet Creek']

import pandas as pd
from sqlalchemy import text

stations = ['Judge Creek', 'Rithet Creek']

SQL = text("""
    SELECT 
        m.station_name,
        COUNT(o.obs_time) AS n_obs,
        MIN(o.obs_time) AS oldest,
        MAX(o.obs_time) AS newest
    FROM meta_history m
    LEFT JOIN obs_raw o ON o.history_id = m.history_id
    WHERE m.station_name = ANY(:stations)
    GROUP BY m.station_name
""")

with engine.connect() as conn:
    df = pd.read_sql(SQL, conn, params={"stations": stations})

print(df)

   station_name  n_obs oldest newest
0   Judge Creek      0   None   None
1  Rithet Creek      0   None   None


In [4]:
import pandas as pd
from sqlalchemy import text

stations = ['Judge Creek', 'Rithet Creek']

SQL_network = text("""
    SELECT 
        h.station_name,
        s.native_id,
        s.station_id,
        s.network_id,
        n.network_name,
        h.lat,
        h.lon
    FROM meta_station s
    JOIN meta_history h ON s.station_id = h.station_id
    JOIN meta_network n ON s.network_id = n.network_id
    WHERE h.station_name = ANY(:stations)
""")

with engine.connect() as conn:
    df_station_network = pd.read_sql(
        SQL_network,
        conn,
        params={"stations": stations}
    )

print(df_station_network)

network_ids = df_station_network["network_id"].unique().tolist()

SQL_vars = text("""
    SELECT DISTINCT
        v.*,
        s.network_id
    FROM meta_vars v
    JOIN meta_station s ON v.network_id = s.network_id
    WHERE v.network_id = ANY(:network_ids)
    ORDER BY s.network_id, v.net_var_name
""")

with engine.connect() as conn:
    df_vars = pd.read_sql(
        SQL_vars,
        conn,
        params={"network_ids": network_ids}
    )

    df_vars


   station_name native_id  station_id  network_id network_name        lat  \
0  Rithet Creek     FW002       12625          36          CRD  48.567500   
1   Judge Creek     HY002       12626          36          CRD  48.585278   

          lon  
0 -123.714167  
1 -123.673611  


In [6]:
SQL_vars = text("""
    SELECT DISTINCT
        v.*
    FROM meta_vars v
    WHERE v.net_var_name = 'WindSpeed'
""")

with engine.connect() as conn:
    df_vars = pd.read_sql(
        SQL_vars,
        conn,
        params={"network_ids": network_ids}
    )

df_vars

,vars_id,network_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name,mod_time,mod_user
0,745,36,kph,None,wind_speed,time: point,None,WindSpeed,Wind Speed,Wspd,2025-02-11 16:03:39.747374,crmp
1,819,5,m s-1,None,wind_speed,time: mean,Wind Speed,WindSpeed,Wind Speed (Mean),Wind Speed,2025-02-11 16:03:39.747374,crmp


#### Insert variable first

In [7]:
TMax = dict(
    network_id=36,
    unit='celsius',
    standard_name='air_temperature',
    cell_method='time: maximum',
    description= 'Maximum hourly temperature',
    name='MAX_TEMP_h',
    display_name='Temperature (Max. h)',
    short_name='air_temperature_maximum_h',
)

TMin = dict(
    network_id=36,
    unit='celsius',
    standard_name='air_temperature',
    cell_method='time: minimum',
    description= 'Minimum hourly temperature',
    name='MIN_TEMP_h',
    display_name='Temperature (Min. h)',
    short_name='air_temperature_minimum_h',
)

RhMax = dict(
    network_id=36,
    unit='percent',
    standard_name='relative_humidity',
    cell_method='time: maximum',
    description=None,
    name='RelativeHumidity_Max',
    display_name='Relative Humidity (Max. h)',
    short_name='Rh_Max',
)

RhMin = dict(
    network_id=36,
    unit='percent',
    standard_name='relative_humidity',
    cell_method='time: minimum',
    description=None,
    name='RelativeHumidity_Min',
    display_name='Relative Humidity (Min. h)',
    short_name='Rh_Min',
)

Rn_1 = dict(
    network_id=36,
    unit='mm',
    standard_name='rainfall_amount',
    cell_method='time: sum',
    description= 'Hourly accumulated rainfall',
    name='Rain_h',
    display_name='Hourly Rainfall',
    short_name='Rn_1',
)

WSpd_1 = dict(
    network_id=36,
    unit='kph',
    standard_name='wind_speed',
    cell_method='time: mean',
    description= 'Hourly average wind speed',
    name='WindSpeed_mean',
    display_name='Wind Speed (mean)',
    short_name='Wspd_mean',
)

WSpd_Max = dict(
    network_id=36,
    unit='kph',
    standard_name='wind_speed',
    cell_method='time: maximum',
    description= 'maximum wind speed (gusts) over the last hour',
    name='WindSpeed_Max',
    display_name='Wind Speed (Max. h)',
    short_name='WSpd_Max',
)

variables_data = [TMax, TMin, RhMax, RhMin, Rn_1, WSpd_1, WSpd_Max]



In [8]:
from pycds import Variable
from sqlalchemy.inspection import inspect

existing = (
    session.query(Variable)
    .filter(
        Variable.network_id == 36,
        Variable.name.in_([v["name"] for v in variables_data]),
    )
    .all()
)

existing_names = {v.name for v in existing}

# Create only the missing ones
to_insert = [
    Variable(**v)
    for v in variables_data
    if v["name"] not in existing_names
]

session.add_all(to_insert)
session.commit()

### Then insert all the obs records

In [9]:
df_station_network

,station_name,native_id,station_id,network_id,network_name,lat,lon
0,Rithet Creek,FW002,12625,36,CRD,48.567500,-123.714167
1,Judge Creek,HY002,12626,36,CRD,48.585278,-123.673611


### generate rows

In [10]:
meta_path = '/workspaces/crmprtd/SSpencerFiles/'

file_names = ['JudgeCreek_raw_MetData_1994-2024.csv', 'RithetCreek_raw_MetData_1995-2014.csv']

import os
import pandas as pd

meta_path = '/workspaces/crmprtd/SSpencerFiles/'
file_names = [
    'JudgeCreek_raw_MetData_1994-2024.csv',
    'RithetCreek_raw_MetData_1995-2014.csv'
]

dfs = []

for file in file_names:
    path = os.path.join(meta_path, file)

    df = pd.read_csv(
        path,
        header=[0, 1]   # read two header rows
    )

    # ---- Combine header levels (e.g. Temp + C → Temp_C)
    df.columns = [
        f"{c1}_{c2}".strip("_")
        for c1, c2 in df.columns
    ]

    df['DateTime_DateTime'] = pd.to_datetime(
        df['DateTime_DateTime'],
        format='%d/%m/%Y %H:%M',
        errors='coerce'
    )


    dfs.append(df)

df_judge = dfs[0]
df_rithet = dfs[1]



In [11]:
df_judge


,StationName_Unnamed: 0_level_1,DateTime_DateTime,Prec_1_mm,Rh_%,RhMax_%,RhMin_%,TMax_C,TMin_C,Temp_C
0,Judge Creek,1994-10-28 15:00:00,0.0,76.0,NaN,NaN,NaN,NaN,9.5
1,Judge Creek,1994-10-28 16:00:00,0.0,76.0,NaN,NaN,NaN,NaN,8.1
2,Judge Creek,1994-10-28 17:00:00,0.0,99.0,NaN,NaN,NaN,NaN,4.1
3,Judge Creek,1994-10-28 18:00:00,0.0,100.0,NaN,NaN,NaN,NaN,1.6
4,Judge Creek,1994-10-28 19:00:00,0.0,100.0,NaN,NaN,NaN,NaN,0.6
...,...,...,...,...,...,...,...,...,...
262970,Judge Creek,2024-12-31 20:00:00,0.0,98.0,98.0,98.0,2.5,2.4,2.4
262971,Judge Creek,2024-12-31 21:00:00,0.0,98.0,98.0,98.0,2.4,2.2,2.2
262972,Judge Creek,2024-12-31 22:00:00,0.0,98.0,98.0,98.0,2.5,2.2,2.5
262973,Judge Creek,2024-12-31 23:00:00,0.0,98.0,98.0,98.0,2.6,2.5,2.6


In [12]:
from sqlalchemy import text
import pandas as pd

SQL_VARS_INFO = text("""
SELECT  *
FROM meta_vars 
WHERE network_id = :network_id
""")

net_vars = pd.read_sql(
    SQL_VARS_INFO,
    engine,
    params={"network_id": 36}
)

net_vars

,vars_id,network_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name,mod_time,mod_user
0,745,36,kph,None,wind_speed,time: point,None,WindSpeed,Wind Speed,Wspd,2025-02-11 16:03:39.747374,crmp
1,744,36,W/m2,None,solar_irradiance,time: point,None,SolarRadiation,Solar Radiation,PYRavg,2025-02-11 16:03:39.747374,crmp
2,742,36,mm,None,lwe_thickness_of_surface_snow_amount,time: point,None,SnowWaterEquivalent,Snow Water Equivalent,SSG_SWE,2025-02-11 16:03:39.747374,crmp
3,743,36,hPa,None,air_pressure,time: point,None,BarometricPressure,Air Pressure,BP,2025-02-11 16:03:39.747374,crmp
4,737,36,mm,None,rainfall_amount,time: sum,None,Rain,Rain,Rn15,2025-02-11 16:03:39.747374,crmp
5,738,36,mm,None,lwe_thickness_of_precipitation_amount,time: sum,None,Precipitation,Precipitation,Prec_1,2025-02-11 16:03:39.747374,crmp
6,740,36,percent,None,relative_humidity,time: point,None,RelativeHumidity,Relative Humidity,Rh,2025-02-11 16:03:39.747374,crmp
7,741,36,m,None,surface_snow_thickness,time: point,None,SnowDepth,Snow Depth,SnowDep,2025-02-11 16:03:39.747374,crmp
8,746,36,degrees,None,wind_from_direction,time: point,None,WindDirection,Wind Direction,Dir,2025-02-11 16:03:39.747374,crmp
9,739,36,celsius,None,air_temperature,time: point,None,AirTemperature,Air temperature,Temp,2025-02-11 16:03:39.747374,crmp


In [14]:
mapping_id = {
    'Prec_1_mm': 738,
    'Rh_%': 740,
    'WSpd_Unnamed: 9_level_1': 745,
    'Temp_C': 739,
    'Dir_deg': 746,

    'TMax_C': 1307,
    'TMin_C': 1308,
    'RhMax_%': 1309,
    'RhMin_%': 1310,
    'Rn_1_mm': 1311,
    'WSpd_1_kph': 1312,
    'Mx_Spd_kph': 1313
}


from sqlalchemy import text

SQL_vars = text("""
    SELECT v.vars_id, v.net_var_name, v.unit
    FROM meta_vars v
    WHERE v.vars_id IN :vars_ids
""")

vars_ids = tuple(mapping_id.values())

with engine.connect() as conn:
    df_vars = pd.read_sql(
        SQL_vars,
        conn,
        params={"vars_ids": vars_ids}
    )

df_mapping = pd.DataFrame(
    list(mapping_id.items()),
    columns=["var_name_raw", "vars_id"]
)

df_var = df_mapping.merge(df_vars, on="vars_id", how="left")

df_var

,var_name_raw,vars_id,net_var_name,unit
0,Prec_1_mm,738,Precipitation,mm
1,Rh_%,740,RelativeHumidity,percent
2,WSpd_Unnamed: 9_level_1,745,WindSpeed,kph
3,Temp_C,739,AirTemperature,celsius
4,Dir_deg,746,WindDirection,degrees
5,TMax_C,1307,MAX_TEMP_h,celsius
6,TMin_C,1308,MIN_TEMP_h,celsius
7,RhMax_%,1309,RelativeHumidity_Max,percent
8,RhMin_%,1310,RelativeHumidity_Min,percent
9,Rn_1_mm,1311,Rain_h,mm


In [15]:
df_rithet

,StationName_Unnamed: 0_level_1,DateTime_DateTime,TMax_C,TMin_C,Temp_C,RhMax_%,RhMin_%,Rh_%,Rn_1_mm,WSpd_Unnamed: 9_level_1,WSpd_1_kph,Dir_deg,Mx_Spd_kph
0,FWx Rithet Creek,1995-11-22 00:00:00,NaN,NaN,NaN,0.0,0.0,0.0,0.0,1.8,NaN,333.0,5.0
1,FWx Rithet Creek,1995-11-22 01:00:00,NaN,NaN,NaN,0.0,0.0,0.0,0.0,1.8,NaN,323.0,3.3
2,FWx Rithet Creek,1995-11-22 02:00:00,NaN,NaN,NaN,0.0,0.0,0.0,0.3,3.2,NaN,164.0,5.6
3,FWx Rithet Creek,1995-11-22 03:00:00,NaN,NaN,NaN,0.0,0.0,0.0,0.0,5.3,NaN,356.0,9.0
4,FWx Rithet Creek,1995-11-22 04:00:00,NaN,NaN,NaN,0.0,0.0,0.0,0.0,3.4,NaN,6.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
160630,FWx Rithet Creek,2014-03-20 20:00:00,2.4,1.4,1.4,94.0,89.0,94.0,0.0,0.0,0.0,2.0,0.0
160631,FWx Rithet Creek,2014-03-20 21:00:00,1.4,0.6,0.6,94.0,93.0,94.0,0.0,0.0,0.0,23.0,0.0
160632,FWx Rithet Creek,2014-03-20 22:00:00,0.6,-0.5,-0.5,95.0,94.0,95.0,0.0,0.0,0.0,16.0,0.0
160633,FWx Rithet Creek,2014-03-20 23:00:00,-0.5,-1.4,-1.4,95.0,95.0,95.0,0.0,0.0,0.0,4.0,0.0


### The first station

In [16]:
from crmprtd import Row

### for the station Rithet Creek

df_station_1 = df_station_network.loc[
    df_station_network['station_name'] == 'Rithet Creek'
]

time_col = df_rithet['DateTime_DateTime']
variable_name = df_var['var_name_raw']
variable_name_db = df_var['net_var_name']
unit_db = df_var['unit']
network_name = df_station_1['network_name']
native_id = df_station_1['native_id']
lat = float(df_station_1['lat'].iloc[0])
lon = float(df_station_1['lon'].iloc[0])

time_col
variable_name


all_rows = []

# station metadata (extract scalar values!)
network_name = df_station_1['network_name'].iloc[0]
native_id = df_station_1['native_id'].iloc[0]
lat = df_station_1['lat'].iloc[0]
lon = df_station_1['lon'].iloc[0]

# loop over each time row
for _, row_i in df_rithet.iterrows():
    
    time_value = row_i['DateTime_DateTime']
    
    # loop over each variable mapping
    for _, var_row in df_var.iterrows():
        
        raw_name = var_row['var_name_raw']
        db_name = var_row['net_var_name']
        unit_db = var_row['unit']
        
        val = row_i.get(raw_name)
        
        if pd.notna(val):   # optional: skip NaNs
            all_rows.append(
                Row(
                    time=time_value,
                    val=val,
                    variable_name=db_name,
                    unit=unit_db,
                    network_name=network_name,
                    station_id=native_id,
                    lat=lat,
                    lon=lon
                )
            )

In [17]:
len(all_rows)

1672856

In [18]:
import sys
sys.path.append("/workspaces/crmprtd/fern/FERNNorth2025_insert")

from auto_insert_func import *

db_url = "postgresql://tongli1997@db.pcic.uvic.ca:5433/crmp?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass"

output_folder = '/workspaces/crmprtd/update_round2/output_tables/'

statin_name = df_station_1['station_name']


safe_insert_rows(
    all_rows,
    log_file_path=output_folder + str(statin_name) +'insert.log',
    db_url=db_url,
    chunk_size=5000,        # SAFE starting point
    bulk_chunk_size=1000,   # internal DB bulk insert size
)

🚀 Starting insert: 1672856 rows in 335 chunks
➡️  Processing rows 0–4999
{"asctime": "2026-02-27 01:02:10,376", "levelname": "INFO", "name": "crmprtd.insert", "message": "Using Bulk Insert Strategy"}
{"asctime": "2026-02-27 01:02:12,451", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 1000 inserted, 0 skipped, 0 failed"}
{"asctime": "2026-02-27 01:02:14,594", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 2000 inserted, 0 skipped, 0 failed"}
{"asctime": "2026-02-27 01:02:16,743", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 3000 inserted, 0 skipped, 0 failed"}
{"asctime": "2026-02-27 01:02:18,817", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 4000 inserted, 0 skipped, 0 failed"}
{"asctime": "2026-02-27 01:02:20,896", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 5000 inserted, 0 skipped, 0 failed"}
{"asctime": "20

### For the second station

In [19]:
from crmprtd import Row

### for the station Rithet Creek

df_station_2 = df_station_network.loc[
    df_station_network['station_name'] == 'Judge Creek'
]

time_col = df_judge['DateTime_DateTime']
variable_name = df_var['var_name_raw']
variable_name_db = df_var['net_var_name']
unit_db = df_var['unit']
network_name = df_station_2['network_name']
native_id = df_station_2['native_id']
lat = float(df_station_2['lat'].iloc[0])
lon = float(df_station_2['lon'].iloc[0])

time_col
variable_name


all_rows = []

# station metadata (extract scalar values!)
network_name = df_station_2['network_name'].iloc[0]
native_id = df_station_2['native_id'].iloc[0]
lat = df_station_2['lat'].iloc[0]
lon = df_station_2['lon'].iloc[0]

# loop over each time row
for _, row_i in df_judge.iterrows():
    
    time_value = row_i['DateTime_DateTime']
    
    # loop over each variable mapping
    for _, var_row in df_var.iterrows():
        
        raw_name = var_row['var_name_raw']
        db_name = var_row['net_var_name']
        unit_db = var_row['unit']
        
        val = row_i.get(raw_name)
        
        if pd.notna(val):   # optional: skip NaNs
            all_rows.append(
                Row(
                    time=time_value,
                    val=val,
                    variable_name=db_name,
                    unit=unit_db,
                    network_name=network_name,
                    station_id=native_id,
                    lat=lat,
                    lon=lon
                )
            )

In [20]:
all_rows

[Row(time=Timestamp('1994-10-28 15:00:00'), val=0.0, variable_name='Precipitation', unit='mm', network_name='CRD', station_id='HY002', lat=np.float64(48.585278), lon=np.float64(-123.673611)),
 Row(time=Timestamp('1994-10-28 15:00:00'), val=76.0, variable_name='RelativeHumidity', unit='percent', network_name='CRD', station_id='HY002', lat=np.float64(48.585278), lon=np.float64(-123.673611)),
 Row(time=Timestamp('1994-10-28 15:00:00'), val=9.5, variable_name='AirTemperature', unit='celsius', network_name='CRD', station_id='HY002', lat=np.float64(48.585278), lon=np.float64(-123.673611)),
 Row(time=Timestamp('1994-10-28 16:00:00'), val=0.0, variable_name='Precipitation', unit='mm', network_name='CRD', station_id='HY002', lat=np.float64(48.585278), lon=np.float64(-123.673611)),
 Row(time=Timestamp('1994-10-28 16:00:00'), val=76.0, variable_name='RelativeHumidity', unit='percent', network_name='CRD', station_id='HY002', lat=np.float64(48.585278), lon=np.float64(-123.673611)),
 Row(time=Timest

In [21]:
import sys
sys.path.append("/workspaces/crmprtd/fern/FERNNorth2025_insert")

from auto_insert_func import *

db_url = "postgresql://tongli1997@db.pcic.uvic.ca:5433/crmp?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass"

output_folder = '/workspaces/crmprtd/update_round2/output_tables/'

statin_name = df_station_2['station_name']


safe_insert_rows(
    all_rows,
    log_file_path=output_folder + str(statin_name) +'insert.log',
    db_url=db_url,
    chunk_size=5000,        # SAFE starting point
    bulk_chunk_size=1000,   # internal DB bulk insert size
)

🚀 Starting insert: 1782308 rows in 357 chunks
➡️  Processing rows 0–4999


{"asctime": "2026-02-27 01:43:17,874", "levelname": "INFO", "name": "crmprtd.insert", "message": "Using Bulk Insert Strategy"}
{"asctime": "2026-02-27 01:43:19,226", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 1000 inserted, 0 skipped, 0 failed"}
{"asctime": "2026-02-27 01:43:20,518", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 2000 inserted, 0 skipped, 0 failed"}
{"asctime": "2026-02-27 01:43:21,813", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 3000 inserted, 0 skipped, 0 failed"}
{"asctime": "2026-02-27 01:43:23,094", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 4000 inserted, 0 skipped, 0 failed"}
{"asctime": "2026-02-27 01:43:24,398", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 5000 inserted, 0 skipped, 0 failed"}
{"asctime": "2026-02-27 01:43:24,400", "levelname": "INFO", "name": "crmprtd.insert", "m